In [ ]:
import gc
import os

import numpy as np
import pandas as pd
import dai
import torch

from transformer_train import MODEL_PATH, BATCH, StockTransformer, build_dataset, load_model


def main(datasources, start_date, end_date):
    table = datasources["bar1m"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        torch.set_num_threads(min(4, os.cpu_count() or 1))
    checkpoint = load_model(MODEL_PATH, map_location=device)
    stats = (np.asarray(checkpoint["mean"], np.float32), np.asarray(checkpoint["std"], np.float32))
    model = StockTransformer(**checkpoint["model_cfg"]).to(device)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()
    shared_gate = checkpoint["model_cfg"].get("model_variant") == "shared_gate"

    members = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    members["date"] = pd.to_datetime(members["date"], errors="coerce").dt.normalize()
    members["instrument"] = members["instrument"].astype(str)
    days = pd.DatetimeIndex(sorted(members["date"].dropna().unique()))
    outputs = []
    with torch.inference_mode():
        for date in days:
            day_members = members[members["date"] == date]
            instruments = day_members["instrument"].drop_duplicates().tolist()
            day_start = date.strftime("%Y-%m-%d 00:00:00")
            day_end = date.strftime("%Y-%m-%d 23:59:59")
            x, _, index, _ = build_dataset(table, day_start, day_end, "infer", instruments, stats)
            scores = []
            if shared_gate:
                whole_day = torch.from_numpy(x).to(device, non_blocking=True)
                gate = model.shared_gate_from_cross_section(whole_day)
                del whole_day
                for start in range(0, len(index), BATCH):
                    block = torch.from_numpy(x[start:start + BATCH]).to(device, non_blocking=True)
                    scores.append(model(block, shared_gate=gate).cpu().numpy())
                    del block
                del gate
            else:
                for start in range(0, len(index), BATCH):
                    block = torch.from_numpy(x[start:start + BATCH]).to(device, non_blocking=True)
                    scores.append(model(block).cpu().numpy())
                    del block
            index["score"] = np.concatenate(scores).astype(np.float64)
            outputs.append(index)
            del x, index, scores, day_members, instruments
            gc.collect()

    prediction = pd.concat(outputs, ignore_index=True)
    result = (prediction.merge(members, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])
        [["date", "instrument", "score"]]
        .reset_index(drop=True))
    return result
